In [191]:
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.callbacks import BaseCallbackHandler
from langchain_groq import ChatGroq
from langchain_core.runnables import RunnablePassthrough
from pydantic import BaseModel
from langchain_classic.retrievers import MultiVectorRetriever

# Used for small2big retrievers
from langchain_classic.retrievers import ParentDocumentRetriever

# InMemory Store for Parent chunks
from langchain_core.stores import InMemoryStore

from langchain_text_splitters import RecursiveCharacterTextSplitter

### Creating a Callback class

In [ ]:
class CallCounter(BaseCallbackHandler):
    def __init__(self):
        self.count=0
    
    def on_chat_model_start(self, *args, **kwargs):
        self.count+=1
    
CALLS=CallCounter()

def make_chat_model(model_name="qwen/qwen3.8-27b"):
    return ChatGroq(
        model=model_name,
        callbacks=[CALLS]
    )

model=make_chat_model()

In [166]:
def to_document(doc, dept, status, text):
    return Document(
        page_content=text,
        metadata={
            "doc": doc,
            "dept": dept,
            "status": status,
        }
    )

In [168]:
CORPUS = [
    to_document("refund_current", "policy", "current", "Refund policy. Annual subscriptions can be refunded within 14 days of purchase. After 14 days we offer store credit for the unused months. Monthly subscriptions can be cancelled at any time, but the current month is not refunded."),
    to_document("faq_refund", "faq", "current", "FAQ: How do refunds work? You can get a refund on an annual subscription within 14 days of buying it. Monthly plans are never refunded for the month already started, but you can cancel any time."),
    to_document("refund_2023", "policy", "archived", "ARCHIVED (2023) refund policy: annual subscriptions could be refunded within 30 days of purchase."),
    to_document("shipping", "policy", "current", "Shipping. Orders of 5 bags or more ship free. Orders placed before 12:00 UTC ship the same day. Standard delivery takes 3 to 5 working days in the UK and 7 to 10 working days elsewhere in Europe."),
    to_document("roasting", "ops", "current", "Roasting schedule. Light roasts are roasted every Tuesday and dark roasts every Friday at our Lisbon roastery. Beans are freshest for 30 days after the roast date when kept in a sealed bag."),
    to_document("blends", "product", "current", "The Drift blend is a light roast from Ethiopia with notes of jasmine and lemon. The Ember blend is a dark roast with notes of cocoa and orange peel."),
    to_document("support", "policy", "current", "Customer support is open 9:00 to 17:00 UTC, Monday to Friday. Premium subscribers get email replies within 4 hours."),
    to_document("giftcards", "policy", "current", "Zorvia gift cards never expire and can be used on subscriptions and single bags."),
    to_document("wholesale", "sales", "current", "Wholesale accounts require a minimum order of 20 kg per month and get 15% off list price."),
    to_document("referral", "marketing", "current", "Referral programme: both the referrer and the friend get one free bag after the friend's first order."),
    to_document("equipment", "product", "current", "Equipment note: the GX-4471 hand grinder is out of stock until March. The GX-5000 electric grinder ships in 2 days."),
    to_document("brewing", "product", "current", "Brewing guide for the Drift blend. Use 15 g of coffee for 250 ml of water at 94 degrees C. Wet the grounds and wait 30 seconds, then pour slowly in circles. Total brew time should be about 3 minutes. If the cup tastes sour, grind finer; if it tastes bitter, grind coarser. Rinse the paper filter first to remove any papery taste."),
    to_document("company", "about", "current", "Zorvia Coffee Co. was founded in Lisbon in 2018 and ships compostable packaging."),
]

print(len(CORPUS), "documents")

13 documents


# Creating Embedding(L2 Normalized) and Text Splitter(Chunking) Instance

In [5]:
embedding=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={
        "normalize_embeddings": True
    }
)

splitter=RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

c:\Users\Shubham Singh\Developer\AIEngineering\pythonAIConcepts\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1789.30it/s]


# Creating a Lambda Function

In [169]:
CURRENT=lambda d: d.metadata.get("status")=="current"

def build_all(docs=None):
    global store, chunks
    chunks=splitter.split_documents(docs or CORPUS)
    store=InMemoryVectorStore(embedding=embedding)
    store.add_documents(chunks)

In [170]:
build_all()

print(len(chunks), "chunks")

14 chunks


### Sample Query to get Embedding of Input Text

In [171]:
vector=embedding.embed_query("What's the refund policy")
print(len(vector), vector[:6])

384 [-0.09544149041175842, 0.04163969308137894, 0.0530204251408577, 0.02047557756304741, 0.06370741128921509, 0.053197115659713745]


# Retrieve the Top K Chunks based on Similarity Score

In [172]:
def retrieve(user_query: str, k=3, filter=None):
    kwargs={"filter": filter} if filter else {}
    return store.similarity_search_with_score(user_query, k, **kwargs)

def show(result):
    for doc, score in result:
        print(f"{score:.3f} [{doc.metadata['status']}] {doc.metadata['doc']}: {doc.page_content[:100]}....")

### 1) No additional Metadata for filtering

In [173]:
QUESTION="Find the onboarding policy and benefits overview."
show(retrieve(QUESTION))

0.189 [current] support: Customer support is open 9:00 to 17:00 UTC, Monday to Friday. Premium subscribers get email replies ....
0.158 [current] referral: Referral programme: both the referrer and the friend get one free bag after the friend's first order....
0.130 [current] wholesale: Wholesale accounts require a minimum order of 20 kg per month and get 15% off list price.....


### 2) Providing additional Metadata for filtering

In [174]:
QUESTION="Whats the refund policy?"
show(retrieve(QUESTION, k=3, filter=CURRENT))

0.592 [current] faq_refund: FAQ: How do refunds work? You can get a refund on an annual subscription within 14 days of buying it....
0.572 [current] refund_current: Refund policy. Annual subscriptions can be refunded within 14 days of purchase. After 14 days we off....
0.282 [current] referral: Referral programme: both the referrer and the friend get one free bag after the friend's first order....


# Naive RAG Pipeline Setup

In [175]:
SYSTEM_PROMPT="""
You are a helpful assistant for Zorvia Coffee Co. Use only the context below. If the context does not contain the answer, say "I don't know".
Keep your answers concise, short and to the point.
"""

# Prompt template that injects retrieved document contents into {context} and the user's query into {question}.
prompt=ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT+"\n\n"+"Context:\n{context}"),
    ("human", "{question}"),
])

# mini-pipeline that takes a completed prompt dictionary containing context and question,
# sends it to the LLM (model), and parses the output into a plain text string (StrOutputParser()).
answer_chain=prompt | model | StrOutputParser()

# Convert the list of top K retrieved Document objects into a single bulleted string.
def format_docs(docs):
    return "\n".join(f"- [{d.metadata['doc']}] {d.page_content}" for d in docs)

# Builds Langchain Expression Language (LCEL) chain.
def make_chain(retriever):
    return {"context": retriever | format_docs, "question": RunnablePassthrough()} | answer_chain


def naive_rag(user_query, k=3, filter=None):
    retriever=store.as_retriever(
        search_kwargs={"k": k, **({"filter": filter} if filter else {})}
    )

    return make_chain(retriever).invoke(user_query)

In [176]:
naive_rag(QUESTION)

'Annual subscriptions can be refunded within 14 days of purchase. After 14 days, we offer store credit for the unused months. Monthly subscriptions can be cancelled at any time, but the current month is not refunded.'

# Creating Small2Big Retriever Instance

In [177]:
small_2_big=ParentDocumentRetriever(
    vectorstore=InMemoryVectorStore(embedding=embedding), # Child/Small Chunks stored in InMemory vector db
    docstore=InMemoryStore(), # Parent/Big chunks stored in InMemory db
    child_splitter=RecursiveCharacterTextSplitter(chunk_size=75, chunk_overlap=10),
    parent_splitter=RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50),
    search_kwargs={"k": 3, **({"filter": CURRENT} if CURRENT else {})},
)

In [178]:
small_2_big.add_documents(CORPUS)

In [179]:
QUESTION_FOR_SMALL2BIG="How fast do premium subscribers get support or expect a reply?"

small_2_big.invoke(QUESTION_FOR_SMALL2BIG)

[Document(metadata={'doc': 'support', 'dept': 'policy', 'status': 'current'}, page_content='Customer support is open 9:00 to 17:00 UTC, Monday to Friday. Premium subscribers get email replies within 4 hours.'),
 Document(metadata={'doc': 'refund_current', 'dept': 'policy', 'status': 'current'}, page_content='Refund policy. Annual subscriptions can be refunded within 14 days of purchase. After 14 days we offer store credit for the unused months. Monthly subscriptions can be cancelled at any time, but the current month is not refunded.')]

In [180]:
print('Parent Chunk: ', small_2_big.invoke(QUESTION_FOR_SMALL2BIG)[0].id, small_2_big.invoke(QUESTION_FOR_SMALL2BIG)[0].page_content)
print('Child Chunk: ', small_2_big.vectorstore.similarity_search(QUESTION_FOR_SMALL2BIG, k=1)[0].metadata.get("doc_id"), small_2_big.vectorstore.similarity_search(QUESTION_FOR_SMALL2BIG, k=1)[0].page_content)

Parent Chunk:  None Customer support is open 9:00 to 17:00 UTC, Monday to Friday. Premium subscribers get email replies within 4 hours.
Child Chunk:  4e1f7159-cb87-4579-97da-85b1b884d115 Premium subscribers get email replies within 4 hours.


# Patch Naive RAG with Small2Big Retriever

In [181]:
def naive_rag_with_small2big(user_query):
    return make_chain(small_2_big).invoke(user_query)

naive_rag_with_small2big(QUESTION_FOR_SMALL2BIG)

'Within 4 hours.'

# Reverse HyDE

In [192]:
class QuestionsForHyDE(BaseModel):
    questions: list[str]

question_creator_chain=(
    ChatPromptTemplate.from_template("Write 3 different questions a customer might ask that this text answers.\nText: {text}")
    | model.with_structured_output(QuestionsForHyDE)
)

In [193]:
current_docs=[c for c in chunks if CURRENT(c)]

questions=question_creator_chain.batch([{"text": c.page_content} for c in current_docs])

In [194]:
questions

[QuestionsForHyDE(questions=['What is the refund policy for annual subscriptions?', 'Can I cancel my monthly subscription and get a refund for the current month?', 'What happens if I want to cancel my annual subscription after 14 days?']),
 QuestionsForHyDE(questions=['What is the refund policy for annual and monthly subscriptions?', 'Can I get a refund if I cancel my monthly plan after starting the month?', 'How many days do I have to request a refund on an annual subscription?']),
 QuestionsForHyDE(questions=['How many bags do I need to order for free shipping?', 'What is the cutoff time for same-day shipping and how long does delivery take?', 'How long does it take for an order to arrive in the UK compared to other parts of Europe?']),
 QuestionsForHyDE(questions=['When do you roast your light and dark roasts at the Lisbon roastery?', 'How long after the roast date do your beans stay freshest?', 'What is the roasting schedule for light and dark roasts at your Lisbon roastery?']),
 Q

In [195]:
chunk_ids=[str(i) for i in range(len(current_docs))]

question_docs=[Document(page_content=question_text, metadata={"chunk_id": chunk_ids[i]}) for i, j in enumerate(questions) for question_text in j.questions]

### Creating Reverse HyDE Instance

In [196]:
reverse_hyde=MultiVectorRetriever(
    vectorstore=InMemoryVectorStore(embedding=embedding), # New questions created are stored in InMemory vector db
    docstore=InMemoryStore(), # Original document to which the questions map stored in InMemory db
    id_key="chunk_id", # this property controls vector stores key to map multiple questions to a same chunk
    search_kwargs={"k": 2},
)

reverse_hyde.vectorstore.add_documents(question_docs)

reverse_hyde.docstore.mset(list(zip(chunk_ids, current_docs)))

QUESTION_FOR_REVERSE_HYDE="How fast do premium subscribers get support or expect a reply?"

print("Parent: ",reverse_hyde.invoke(QUESTION_FOR_REVERSE_HYDE))

Parent:  [Document(metadata={'doc': 'support', 'dept': 'policy', 'status': 'current'}, page_content='Customer support is open 9:00 to 17:00 UTC, Monday to Friday. Premium subscribers get email replies within 4 hours.')]
